# Set Environment & Globals

In [ ]:
# base
import pickle
import os
from pathlib import Path
import warnings
import logging

logging.basicConfig(level="WARNING")
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

# data manipulation
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import itables

itables.init_notebook_mode(all_interactive=True)

warnings.simplefilter("ignore", pd.errors.DtypeWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.WARNING)

# single cell
import scanpy as sc
import squidpy as sq
import spatialdata as sd
import spatialdata_plot as sdp
import spatialdata_io

sc.settings.verbosity = 0

# custom
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from spatial_seq.plot import *
from utils import *

# R
import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter

converter = get_converter()
%load_ext rpy2.ipython

# analysis & device specific
%matplotlib inline
CORES = 10
DATADIR = Path("../../../data")
REFDIR = Path("../../../references")
mpl.rcdefaults()

# Access SpatialData

In [ ]:
def read_visium_hd_segmented(
    outs_path: str,
    sample_id: str,
    shapes_name: str = "cell_segmentations",
) -> sd.models.TableModel:
    outs_path = Path(outs_path)
    bin_size = "segmented_outputs"
    path_bin = outs_path / bin_size
    path_bin_spatial = path_bin / VisiumHDKeys.SPATIAL

    # Load gene expression
    counts_file = "raw_feature_cell_matrix.h5"
    adata = sc.read_10x_h5(path_bin / counts_file, gex_only=False)
    adata.var_names_make_unique()

    # Load scalefactors and images
    with open(path_bin_spatial / VisiumHDKeys.SCALEFACTORS_FILE) as f:
        scalefactors = json.load(f)

    hires_img = np.array(Image.open(path_bin_spatial / "tissue_hires_image.png"))
    lowres_img = np.array(Image.open(path_bin_spatial / "tissue_lowres_image.png"))

    library_id = sample_id
    adata.uns["spatial"] = {
        library_id: {
            "images": {
                "hires": hires_img,
                "lowres": lowres_img,
            },
            "scalefactors": scalefactors,
            "metadata": {"source_image_path": "tissue_hires_image.png"},
        }
    }

    # Assign region info
    adata.obs[VisiumHDKeys.INSTANCE_KEY] = np.arange(len(adata))
    adata.obs[VisiumHDKeys.REGION_KEY] = shapes_name
    adata.obs[VisiumHDKeys.REGION_KEY] = adata.obs[VisiumHDKeys.REGION_KEY].astype(
        "category"
    )

    # Load cell shapes
    shapes = spatialdata_io.geojson(
        outs_path / "segmented_outputs" / f"{shapes_name}.geojson",
        coordinate_system=sample_id,
    )

    # Match shape order to adata
    centroids = shapes.geometry.centroid
    adata.obsm["spatial"] = np.vstack([centroids.x.values, centroids.y.values]).T

    # Estimate spot diameter
    areas = shapes.geometry.area
    mean_area = np.mean(areas)
    diameter_pixels = 2 * np.sqrt(mean_area / np.pi)
    adata.uns["spatial"][library_id]["scalefactors"][
        "spot_diameter_fullres"
    ] = diameter_pixels

    return sd.models.TableModel.parse(
        adata=adata,
        region=shapes_name,
        region_key=str(VisiumHDKeys.REGION_KEY),
        instance_key=str(VisiumHDKeys.INSTANCE_KEY),
    )


def GetSpatialData(
    directory: Path,
    savedir: Path,
    intermediate_path: str = "outs",
    exclude: Iterable = [],
    from_scratch: bool = True,
    overwrite: bool = False,
    get_custom_barcodes: bool = False,
    **kwargs,
):
    """
    Gets all spatial data from a directory.
    `**kwargs` are sent to `spatialdata_io.visium_hd()`
    Run sdatas = sd.concatenate(sdatas, concatenate_tables=True) to concatenate
    """
    sdatas = {}
    barcodes = {}
    samples = os.listdir(directory)
    for s in exclude:
        samples.remove(s)

    for n, sample in tqdm(enumerate(samples)):
        savefile = savedir / sample / "object.zarr"
        outs_dir = directory / sample / intermediate_path

        if from_scratch is True:
            sdata = spatialdata_io.visium_hd(
                outs_dir,
                dataset_id=sample,
                load_all_images=True,
                var_names_make_unique=True,
                **kwargs,
            )

            # rename shapes
            for img_name in ["cytassist_image", "hires_image", "lowres_image"]:
                sdata[img_name] = sdata[f"{sample}_{img_name}"]
                del sdata[f"{sample}_{img_name}"]

            # rename shapes & annotations
            for bin_size in ["002", "008", "016"]:
                sdata[f"SHAPE_square_{bin_size}um"] = sdata[
                    f"{sample}_square_{bin_size}um"
                ]
                del sdata[f"{sample}_square_{bin_size}um"]
                sdata[f"square_{bin_size}um"].uns["spatialdata_attrs"]["region"] = [
                    f"SHAPE_square_{bin_size}um"
                ]
                sdata[f"square_{bin_size}um"].obs[
                    "region"
                ] = f"SHAPE_square_{bin_size}um"

            # add segmentation information
            sdata["nucleus_segmentation"] = spatialdata_io.geojson(
                outs_dir / "segmented_outputs" / "nucleus_segmentations.geojson",
                coordinate_system=sample,
            ).rename_axis("location_id")

            sdata["cell_segmentation"] = spatialdata_io.geojson(
                outs_dir / "segmented_outputs" / "cell_segmentations.geojson",
                coordinate_system=sample,
            ).rename_axis("location_id")

            sdata[f"segmented_bins"] = read_visium_hd_segmented(
                outs_dir,
                sample_id=sample,
                shapes_name="cell_segmentations",
            )

            for table in sdata.tables.values():
                table.obs["Identifier"] = sample
                table.layers["counts"] = table.X

            if overwrite is True:
                sdata.write(savefile)
                sdata = sd.read_zarr(savefile)
        else:
            sdata = sd.read_zarr(savefile)

        sdatas[sample] = sdata

        if get_custom_barcodes is True:
            spatial_barcode_subsets = savedir / sample
            barcodes[sample] = {}
            for csv in spatial_barcode_subsets:
                if ".csv" in csv:
                    barcodes[sample][csv] = pd.read_csv(savedir / sample / csv)
                    barcodes[sample][csv]["Barcode"] = (
                        barcodes[sample][csv]["Barcode"] + f"-{sample}"
                    )
                    barcodes[sample][csv].set_index("Barcode", inplace=True)

    return samples, sdatas, barcodes

In [ ]:
# WRITE PROCESSED
samples, sdatas, barcodes = GetSpatialData(
    DATADIR / "spaceranger" / "LM13969",
    DATADIR / "processed" / "spatial" / "raws",
    exclude=["LFD-cKO-male-780-VAT-2"],
    overwrite=False,
)

sdata = sd.concatenate(sdatas, concatenate_tables=True)
sdata.write(DATADIR / "processed" / "spatial" / "raws" / "COMBINED", overwrite=True)
spatial_uns = {}  # fix spatial uns merging
for s in sdatas:
    for k, v in sdatas[s]["segmented_bins"].uns["spatial"].items():
        spatial_uns.setdefault(k, []).append(v)
with open(DATADIR / "processed" / "spatial" / "raws" / "spatial_uns.pkl", "wb") as file:
    pickle.dump(spatial_uns, file)

In [ ]:
# LOAD PROCESSED
sdata = sd.read_zarr(DATADIR / "processed" / "spatial" / "raws" / "COMBINED")
samples = list(sdata["square_008um"].obs["Identifier"].unique())

### Import data

In [ ]:
# rename to create consistent AnnData .obs names
for csv in barcodes:
    if csv == "JB autocluster.csv":
        newname = "Autocluster Labels"
    elif csv == "JB drawn.csv":
        newname = "ROI Labels"
    elif csv == "EH whole tissue.csv":
        newname = "Whole Tissue"
    else:
        print(csv)

    colname = barcodes[sample][csv].columns[-1]
    barcodes[sample][csv].rename(columns={colname: newname}, inplace=True)

# custom table processing
adatas = []

for samp in samples:
    adata = sdata["square_008um"][sdata["square_008um"].obs["Identifier"] == samp]

    # whole tissue barcodes
    adata.obs["in EH whole tissue"] = adata.obs_names.isin(
        barcodes[samp]["EH whole tissue.csv"].index
    )

    # autocluster barcodes
    adata.obs["in JB autocluster"] = adata.obs_names.isin(
        barcodes[samp]["JB autocluster.csv"].index
    )
    adata.obs = adata.obs.join(barcodes[samp]["JB autocluster.csv"], how="left")
    adata.obs["Autocluster Labels"] = adata.obs["Autocluster Labels"].fillna(
        "not present"
    )
    adata.obs["Autocluster Labels"] = adata.obs["Autocluster Labels"].astype(str)

    # drawn ROIs
    adata.obs["in JB drawn"] = adata.obs_names.isin(
        barcodes[samp]["JB drawn.csv"].index
    )
    adata.obs = adata.obs.join(barcodes[samp]["JB drawn.csv"], how="left")
    adata.obs["ROI Labels"] = adata.obs["ROI Labels"].fillna("not present")
    adata.obs["ROI Labels"] = adata.obs["ROI Labels"].astype(str)

    adatas.append(adata)

adata = sc.concat(adatas)

for col in [
    "in EH whole tissue",
    "in JB autocluster",
    "Autocluster Labels",
    "in JB drawn",
    "ROI Labels",
]:
    sdata["square_008um"].obs[clean_string(col)] = adata.obs[col]

# other metadata
idents = sdata["square_008um"].obs["Identifier"].str.split("-|_", expand=True)
sdata["square_008um"].obs["Diet"] = idents[0]
sdata["square_008um"].obs["Model"] = idents[1]

In [ ]:
%%time
# get aligned hires/lowres images

f, axs = plt.subplots(
    2, len(samples), figsize=(len(samples) * 5, 10), layout="constrained"
)
for i, samp in enumerate(samples):
    # get extents
    adata = sdata["square_008um"]
    data_extent = {}
    mins = np.min(adata[adata.obs["Identifier"] == samp].obsm["spatial"], axis=0)
    maxs = np.max(adata[adata.obs["Identifier"] == samp].obsm["spatial"], axis=0)
    data_extent["x"] = (mins[0], maxs[0])
    data_extent["y"] = (mins[1], maxs[1])

    # hires
    queried_img = bounding_box_query(
        sdata[f"hires_image-{samp}"],
        min_coordinate=[data_extent["x"][0], data_extent["y"][0]],
        max_coordinate=[data_extent["x"][1], data_extent["y"][1]],
        axes=("x", "y"),
        target_coordinate_system=samp,
    )
    sdata[f"queried_hires-{samp}"] = queried_img
    sdata.pl.render_images(f"queried_hires-{samp}").pl.show(
        coordinate_systems=f"{samp}_downscaled_hires", ax=axs[0][i]
    )

    # lowres
    queried_img = bounding_box_query(
        sdata[f"lowres_image-{samp}"],
        min_coordinate=[data_extent["x"][0], data_extent["y"][0]],
        max_coordinate=[data_extent["x"][1], data_extent["y"][1]],
        axes=("x", "y"),
        target_coordinate_system=samp,
    )
    sdata[f"queried_lowres-{samp}"] = queried_img
    sdata.pl.render_images(f"queried_lowres-{samp}").pl.show(
        coordinate_systems=f"{samp}_downscaled_lowres", ax=axs[1][i]
    )

In [ ]:
# SAVE
sdata.write(os.path.join(DATADIR, "processed", "raws", "COMBINED_RAW"))

### Sample Plot

# Preprocessing (standard)

In [ ]:
# LOAD
adata_FULL_WT = sc.read_h5ad(
    os.path.join(DATADIR, "processed", "combined", "raw_processed-V5 WT_filter70.h5ad")
)

### QC

In [ ]:
for m, binsize in enumerate(["square_002um", "square_008um", "square_016um"]):
    # calculated QCs
    sc.pp.calculate_qc_metrics(sdata[binsize], inplace=True, percent_top=[20])
    Filter_GeneGroup(sdata[binsize])

    # f, axs = plt.subplots(3, len(samples), figsize=(22, 18), layout="constrained")
    # for n, samp in enumerate(samples):
    #     adata = sdata[binsize][sdata[binsize].obs["Identifier"] == samp]

    #     # Plot QCs
    #     axs[m][n].set_xlim(right=adata.obs["n_genes_by_counts"].quantile(0.05) * 1.1)
    #     sns.histplot(adata.obs["n_genes_by_counts"], ax=axs[m][n], stat="proportion")
    #     plt.axvline(np.mean(adata.obs["n_genes_by_counts"]), color="red")

    # sc.pl.violin(
    #     adata,
    #     ["log1p_n_genes_by_counts", "log1p_total_counts", "pct_counts_mito"],
    #     jitter=0.4,
    #     multi_panel=True,
    # )

    groupby = "Identifier"
    # check_QCPlot(sdata[binsize].obs, "log1p_total_counts", groupby)
    # check_QCPlot(sdata[binsize].obs, "pct_counts_mito", groupby)
    # check_QCPlot(sdata[binsize].obs, "n_genes", groupby)
    check_QCPlot(sdata[binsize].obs, "pct_counts_in_top_20_genes", groupby)

In [ ]:
%%time

# View images
f, axs = plt.subplots(
    1, len(samples), figsize=(len(samples) * 5, 5), layout="constrained"
)
for i, samp in enumerate(samples):
    sdatas[samp].pl.render_images("queried_lowres").pl.show(
        coordinate_systems=samp, ax=axs[i]
    )

In [ ]:
%%time
gene_name = "Igkc"
new_cmap = set_zero_in_cmap_to_transparent(cmap="viridis")

f, axs = plt.subplots(
    1, len(samples), figsize=(len(samples) * 5, 5), layout="constrained"
)
for i, samp in enumerate(samples):
    sdatas[samp].pl.render_shapes(
        f"{samp}_square_008um",
        color=gene_name,
        cmap=new_cmap,
    ).pl.show(coordinate_systems=samp, ax=axs[i])

In [ ]:
runFilters = True
adatas = {}

# QC Thresholds
filters = {"GenesPerCell": 60, "ReadsPerCell": 0, "CellsPerGene": 10, "Mito%": 15}

if runFilters is True:
    for s in samples:
        adata = sdata["square_008um"][
            sdata["square_008um"].obs["Identifier"] == s
        ].copy()

        # Filter on whole tissue
        removed = np.sum(adata.obs["in_EH_whole_tissue"] == False)
        print(f"Whole Tissue Filter: {removed} ({removed / adata.shape[0] * 100:.2f}%)")
        adata = adata[adata.obs["in_EH_whole_tissue"]]

        # Filter out annoying IgG genes
        adata = adata[:, adata.var_names.str.contains("Igh|Igk|Jchain") == 0]

        # Standard QC Filters
        Filter_QC(
            adata,
            GenePerCell=filters["GenesPerCell"],
            CountPerCell=filters["ReadsPerCell"],
            CellPerGene=filters["CellsPerGene"],
            verbose=True,
        )
        Filter_GeneGroup(
            adata,
            key="mito",
            marker="mt",
            verbose=True,
            perc_threshold=filters["Mito%"],
        )

        print(f"Remaining bins: {adata.shape[0]}\n")
        adatas[s] = adata.copy()

In [ ]:
# plot target genes upon images
new_cmap = set_zero_in_cmap_to_transparent(cmap="viridis")

f, axs = plt.subplots(
    1, len(samples), figsize=(len(samples) * 5, 5), layout="constrained"
)
for i, samp in enumerate(samples):
    sdata.pl.render_images(f"queried_hires-{samp}").pl.render_shapes(
        f"SHAPE_square_008um-{samp}",
        color="Cma1",
        cmap=new_cmap,
    ).pl.show(coordinate_systems=f"{samp}_downscaled_hires", ax=axs[i])

In [ ]:
# QC check 2
# adata = sdatas[samp].tables["square_008um"].copy()
for samp in samples:
    adata = adatas[samp].copy()

    # calculated QCs
    sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=[20])
    Filter_GeneGroup(adata)

    # plot QCs
    sc.pl.violin(
        adata,
        ["log1p_n_genes_by_counts", "log1p_total_counts", "pct_counts_mito"],
        jitter=0.4,
        multi_panel=True,
    )

    f, axs = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
    sns.histplot(
        adata.obs["total_counts"],
        kde=True,
        ax=axs[0],
        stat="proportion",
        # log_scale=True,
    )
    axs[0].set_xlim(0, 200)

    sns.histplot(
        adata.obs["n_genes_by_counts"],
        kde=True,
        ax=axs[1],
        stat="proportion",
        # log_scale=True,
    )
    axs[1].set_xlim(0, 200)

In [ ]:
# View unintegrated version

adata_unInt = sdata["square_008um"].copy()
adata_unInt.uns["QC filters"] = filters

# order categories
order_obs(adata_unInt, "Diet", ["LFD", "HFD"])
order_obs(adata_unInt, "Model", ["CTR", "cKO"])

Normalize(adata_unInt, kind="log1p")
FindVariableGenes(
    adata_unInt, kind="seurat_v3", batch_column="Identifier", n_features=100
)

sc.pp.neighbors(adata_unInt)
sc.tl.umap(adata_unInt, key_added=f"UMAP_unintegrated")
lm = LocalMAP()
adata.obsm[f"LocalMAP_unintegrated"] = lm.fit_transform(adata.X.toarray())

plotting = ["Identifier", "log1p_total_counts"]
sc.pl.embedding(adata, "LocalMAP_unintegrated", color=plotting)
sc.pl.embedding(adata, "UMAP_unintegrated", color=plotting)

### Integration

In [ ]:
batch_column = "Identifier"
hvg_kind = "seurat_v3"
int_kind = "harmony"

adata_FULL_WT = sc.concat(
    [adatas[samp] for samp in samples],
    join="outer",
)
adata_FULL_WT.uns["QC filters"] = filters
adata_FULL_WT.uns["spatial_attrs"] = sdata["square_008um"].uns["spatial_attrs"]

# order categories
order_obs(adata_FULL_WT, "Diet", ["LFD", "HFD"])
order_obs(adata_FULL_WT, "Model", ["CTR", "cKO"])

Normalize(adata_FULL_WT, kind="log1p")
FindVariableGenes(
    adata_FULL_WT, kind=hvg_kind, batch_column=batch_column, n_features=500
)
print(adata_FULL_WT.var_names[adata_FULL_WT.var["highly_variable"]].to_list(), "\n")

Integrate(adata_FULL_WT, kind=int_kind, batch_column=batch_column, use_var_genes=False)
Visualize(adata_FULL_WT)

In [ ]:
# SAVE
adata_FULL_WT.write(
    os.path.join(
        DATADIR,
        "processed",
        "spatial",
        "combined",
        "raw_processed-V2.2 WT_filter60.h5ad",
    )
)

# Segmented inputs

## SD approach

In [ ]:
# LOAD PROCESSED
sdata = sd.read_zarr(DATADIR / "processed" / "spatial" / "raws" / "COMBINED")
samples = list(sdata["square_008um"].obs["Identifier"].unique())

#### QC & Checks

In [ ]:
# calculated QCs
sc.pp.calculate_qc_metrics(sdata["segmented_bins"], inplace=True, percent_top=[20])
Filter_GeneGroup(sdata["segmented_bins"])

metadata = sdata["segmented_bins"].obs
groupby = "Identifier"
check_QCPlot(metadata, "log1p_total_counts", groupby)
check_QCPlot(metadata, "n_genes_by_counts", groupby)
check_QCPlot(metadata, "pct_counts_in_top_20_genes", groupby)

metadata

In [ ]:
from spatialdata_plot.pl.utils import set_zero_in_cmap_to_transparent

aggregated = sdata.aggregate(
    values=f"SHAPE_square_002um-{samples[0]}",
    by=f"nucleus_segmentation-{samples[0]}",
    table_name="square_002um",
    value_key="Igkc",
    target_coordinate_system=samples[0],
)
new_cmap = set_zero_in_cmap_to_transparent(cmap="viridis")

aggregated.pl.render_shapes(
    f"nucleus_segmentation-{samples[0]}", color="Igkc", cmap=new_cmap, vmax=100
).pl.show()

In [ ]:
sample = samples[0]

xs = [19000, 21000]
ys = [17000, 19000]

crop_sdata1 = sd.bounding_box_query(
    sdata,
    axes=("y", "x"),
    min_coordinate=[ys[0], xs[0]],
    max_coordinate=[ys[1], xs[1]],
    target_coordinate_system=samples[0],
)

fig, axs = plt.subplots(3, 2, figsize=(12, 16), layout="constrained")
axs = axs.flatten()

sdata.pl.render_images(f"hires_image-{sample}").pl.show(
    coordinate_systems=sample, ax=axs[0]
)
axs[0].add_patch(
    plt.Rectangle(
        (xs[0], ys[0]),
        xs[1] - xs[0],
        ys[1] - ys[0],
        ls="--",
        ec="Red",
        fc="Grey",
        alpha=0.5,
    )
)


crop_sdata1.pl.render_images(f"hires_image-{sample}").pl.show(
    coordinate_systems=sample, ax=axs[1], title="Cropped area"
)

crop_sdata1.pl.render_images(f"hires_image-{sample}").pl.render_shapes(
    f"nucleus_segmentation-{sample}",
    fill_alpha=0.5,
    na_color="blue",
    outline_width=0.3,
    outline_color="black",
).pl.show(coordinate_systems=sample, ax=axs[2], title="Nuclear segmentation")

crop_sdata1.pl.render_images(f"hires_image-{sample}").pl.render_shapes(
    f"cell_segmentation-{sample}",
    fill_alpha=0.5,
    na_color="blue",
    outline_width=0.3,
    outline_color="black",
).pl.show(coordinate_systems=sample, ax=axs[3], title="Cell segmentation")

crop_sdata1.pl.render_shapes(
    f"cell_segmentation-{sample}",
    color="Igkc",
    outline_width=0.3,
    outline_color="black",
).pl.show(coordinate_systems=sample, ax=axs[4], title="Igkc expression")
crop_sdata1.pl.render_shapes(
    f"cell_segmentation-{sample}",
    color="Adipoq",
    outline_width=0.3,
    outline_color="black",
).pl.show(coordinate_systems=sample, ax=axs[5], title="Adipoq expression")

# crop_sdata1.pl.render_shapes(f"cell_segmentation-{sample}", color="Mmp12").pl.show(
#     coordinate_systems=sample,
#     ax=axs[5],
# )

#### Initial Analysis

In [ ]:
filters = {"GenesPerCell": 0, "ReadsPerCell": 30, "CellsPerGene": 10, "Mito%": 15}

adata = sdata["segmented_bins"].copy()

# clean mitochondrial genes
adata.var["mito"] = adata.var_names.str.startswith("mt-")
adata.obsm["mito"] = adata[:, adata.var["mito"].values].X.toarray()
adata = adata[:, ~adata.var["mito"].values]

# clean Ig genes
adata.var["igs"] = adata.var_names.str.contains("Igh|Igk|Jchain")
adata.obsm["igs"] = adata[:, adata.var["igs"].values].X.toarray()
adata = adata[:, ~adata.var["igs"].values]

Filter_QC(
    adata,
    GenePerCell=filters["GenesPerCell"],
    CountPerCell=filters["ReadsPerCell"],
    CellPerGene=filters["CellsPerGene"],
    verbose=True,
)
Filter_GeneGroup(
    adata,
    key="mito",
    marker="mt",
    verbose=True,
    perc_threshold=filters["Mito%"],
)

print(f"Remaining bins: {adata.shape[0]}\n")

groupby = "Identifier"
check_QCPlot(adata.obs, "log1p_total_counts", groupby)
check_QCPlot(adata.obs, "n_genes_by_counts", groupby)
check_QCPlot(adata.obs, "pct_counts_in_top_20_genes", groupby)
print(f"mean UMIs: {np.mean(adata.obs['n_genes_by_counts']):.2f}")

adata

In [ ]:
np.random.seed(123)

batch_column = "Identifier"
adata.layers["counts"] = adata.X.copy()
Normalize(adata)
FindVariableGenes(adata, "seurat_v3_paper", batch_column, 2000)
Integrate(adata, batch_column, True)
Visualize(adata, localmap=True)
sc.pl.embedding(adata, "UMAP")
sc.pl.embedding(adata, "LocalMAP")

In [ ]:
tmp = adata[adata.obs["Identifier"] == samples[0]]
sc.pl.spatial(
    tmp,
    color=["log1p_total_counts", "pct_counts_in_top_20_genes"],
    spot_size=100,
    cmap="Reds",
)

In [ ]:
resolutions = [0.1, 0.5, 1.0, 1.5]
Cluster(adata, resolutions=resolutions)

In [ ]:
sc.pl.embedding(
    adata,
    "UMAP",
    color=["log1p_total_counts", "pct_counts_in_top_20_genes", "Identifier"]
    + [f"leiden_{res}" for res in resolutions],
)
sc.pl.embedding(adata, "UMAP", color=["leiden_0.5"], legend_loc="on data")
sc.pl.violin(adata, ["log1p_total_counts"], "leiden_0.5")

In [ ]:
clusters = "leiden_0.5"
de_key = "DEGs"
runDEGs = False

if runDEGs is True:
    sc.tl.rank_genes_groups(
        adata,
        groupby=clusters,
        key_added=de_key,
        use_raw=False,
        layer="normalized",
        method="wilcoxon",
    )

sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby=clusters,
    key=de_key,
    # standard_scale="var",
    n_genes=5,
    # min_logfoldchange=2,
    # ax=ax,
)

sc.pl.rank_genes_groups_heatmap(adata, key=de_key, groupby=clusters, layer="normalized")

sc.pl.violin(adata, "n_genes", groupby=clusters)

In [ ]:
sigs = pd.concat(
    [
        pd.read_csv(REFDIR / "custom_signatures" / sig, index_col=0)
        for sig in os.listdir(REFDIR / "custom_signatures")
    ]
)

for group, df in sigs.groupby("group"):
    sc.tl.score_genes(adata, df.names, score_name=f"{group}_scores")

In [ ]:
def concat_spatial(x, y, ref_x, ref_y, offset_x=1, offset_y=0, mode="perc"):
    """
    Offsets starting from bottom right corner (Xmax, Ymin).
    Default offset is 1 full width of reference to the right of ref_x.
    """

    if mode == "perc":
        offset_x *= ref_x.max() - ref_x.min()
        offset_y *= ref_y.max() - ref_y.min()

    return x - x.min() + ref_x.max() + offset_x, y - y.min() + ref_y.min() + offset_y


samples = adata.obs["Identifier"].unique()
adata.obs["x"], adata.obs["y"] = np.hsplit(adata.obsm["spatial"], 2)

ref = samples[0]
for n in np.arange(1, len(samples)):
    ref = samples[n - 1]
    samp = samples[n]
    ref_adata = adata[adata.obs["Identifier"] == ref].copy()
    new_adata = adata[adata.obs["Identifier"] == samp].copy()

    adata.obs["x"].loc[new_adata.obs_names], adata.obs["y"].loc[new_adata.obs_names] = (
        concat_spatial(
            new_adata.obs["x"],
            new_adata.obs["y"],
            ref_adata.obs["x"],
            ref_adata.obs["y"],
            offset_x=0.5,
        )
    )

adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy()

In [ ]:
import glasbey

palette = glasbey.create_palette(palette_size=len(adata.obs["leiden_0.5"].unique()))

f, ax = plt.subplots(1, 1, figsize=(18, 8), layout="constrained")
sc.pl.spatial(adata, spot_size=150, color="Identifier", ax=ax, frameon=False)

f, ax = plt.subplots(1, 1, figsize=(18, 8), layout="constrained")
sc.pl.spatial(
    adata, spot_size=150, color="leiden_0.5", ax=ax, frameon=False, palette=palette
)

In [ ]:
clustering = "leiden_0.5"

category_scores = ["Immune_scores", "Stromal_scores"]
celltype_scores = adata.obs.columns[-12:][
    ~adata.obs.columns[-12:].isin(["Immune_scores", "Stromal_scores"])
].to_list()

score_adata = sc.AnnData(
    X=adata.obs[adata.obs.columns[-12:]], obs=adata.obs[adata.obs.columns[-16:-12]]
)

sc.pl.matrixplot(
    score_adata,
    var_names=category_scores + celltype_scores,
    groupby=clustering,
    dendrogram=True,
    cmap="RdBu_r",
    vcenter=0,
)

f, ax = plt.subplots(1, 1, figsize=(10, 10), layout="constrained")
sc.pl.embedding(adata, "UMAP", color=clustering, legend_loc="on data", ax=ax)

##### Save/Load

In [ ]:
os.makedirs(DATADIR / "processed" / "spatial" / "analysis", exist_ok=True)
adata.write(DATADIR / "processed" / "spatial" / "analysis" / "30_10-clustered.h5ad")

In [ ]:
adata = sc.read_h5ad(
    DATADIR / "processed" / "spatial" / "analysis" / "30_10-clustered.h5ad"
)
samples = list(adata.obs["Identifier"].unique())

#### Split low & high quality

In [ ]:
adata.obs["high quality"] = adata.obs["leiden_0.1"].isin(np.arange(0, 1).astype(str))
tmp = adata[adata.obs["Identifier"] == samples[2]]
sc.pl.spatial(tmp, color=["high quality"], spot_size=150)

In [ ]:
lq_adata = adata[adata.obs["high quality"] == False]
del lq_adata.uns, lq_adata.obsm, lq_adata.varm, lq_adata.obsp
hq_adata = adata[adata.obs["high quality"] == True]
print(lq_adata, hq_adata)

In [ ]:
del lq_adata.uns["methods"]

In [ ]:
batch_column = "Identifier"

FindVariableGenes(hq_adata, "seurat_v3_paper", batch_column, 2000)
Integrate(hq_adata, batch_column, False)
Visualize(hq_adata, localmap=True)

Integrate(lq_adata, batch_column, False)
Visualize(lq_adata, localmap=True)

##### Cluster

In [ ]:
resolutions = [0.1, 0.5, 1.0, 1.5]
Cluster(lq_adata, resolutions=[0.1, 0.5, 1.0, 1.5])
Cluster(hq_adata, resolutions=[0.1, 0.5, 1.0, 1.5, 2.0, 2.5])

In [ ]:
sc.pl.embedding(
    lq_adata,
    "UMAP",
    color=["Identifier", "log1p_total_counts", "pct_counts_in_top_20_genes"]
    + [f"leiden_{res}" for res in [0.1, 0.5, 1.0, 1.5]],
    frameon=False,
)

sc.pl.embedding(
    hq_adata,
    "UMAP",
    color=["Identifier", "log1p_total_counts", "pct_counts_in_top_20_genes"]
    + [f"leiden_{res}" for res in [0.1, 0.5, 1.0, 1.5, 2.0, 2.5]],
    frameon=False,
)

##### Score

In [ ]:
sigs = pd.concat(
    [
        pd.read_csv(REFDIR / "custom_signatures" / sig, index_col=0)
        for sig in os.listdir(REFDIR / "custom_signatures")
    ]
)

for group, df in sigs.groupby("group"):
    sc.tl.score_genes(hq_adata, df.names, score_name=f"{group}_scores")

all_scores = hq_adata.obs.columns[hq_adata.obs.columns.str.contains("scores")]
category_scores = ["Immune_scores", "Stromal_scores"]
celltype_scores = all_scores[~all_scores.isin(category_scores)].to_list()
all_scores = category_scores + celltype_scores
leiden_cols = hq_adata.obs.columns[hq_adata.obs.columns.str.contains("leiden")]

score_adata = sc.AnnData(
    X=hq_adata.obs[all_scores],
    obs=hq_adata.obs[leiden_cols],
)

In [ ]:
clustering = "leiden_2.5"

sc.pl.matrixplot(
    score_adata,
    var_names=all_scores,
    groupby=clustering,
    dendrogram=True,
    cmap="RdBu_r",
    vcenter=0,
)

f, ax = plt.subplots(1, 1, figsize=(7, 7), layout="constrained")
sc.pl.embedding(hq_adata, "UMAP", color=clustering, legend_loc="on data", ax=ax)

##### DEGs

In [ ]:
clusters = "leiden_1.5"
de_key = "DEGs"
runDEGs = True

if runDEGs is True:
    sc.tl.rank_genes_groups(
        lq_adata,
        groupby=clusters,
        key_added=de_key,
        use_raw=False,
        layer="normalized",
        method="wilcoxon",
    )

sc.pl.rank_genes_groups_dotplot(
    lq_adata,
    groupby=clusters,
    key=de_key,
    # standard_scale="var",
    n_genes=5,
    # min_logfoldchange=2,
    # ax=ax,
)

sc.pl.rank_genes_groups_heatmap(
    lq_adata, key=de_key, groupby=[clusters], layer="normalized", cmap="Reds"
)

In [ ]:
clusters = "leiden_2.5"
de_key = "DEGs"
runDEGs = True

if runDEGs is True:
    sc.tl.rank_genes_groups(
        hq_adata,
        groupby=clusters,
        key_added=de_key,
        use_raw=False,
        layer="normalized",
        method="wilcoxon",
    )

sc.pl.rank_genes_groups_dotplot(
    hq_adata,
    groupby=clusters,
    key=de_key,
    # standard_scale="var",
    n_genes=5,
    # min_logfoldchange=2,
    # ax=ax,
)

sc.pl.rank_genes_groups_heatmap(
    hq_adata, key=de_key, groupby=[clusters], layer="normalized", cmap="Reds"
)

In [ ]:
os.makedirs(DATADIR / "processed" / "spatial" / "analysis", exist_ok=True)
lq_adata.write(
    DATADIR / "processed" / "spatial" / "analysis" / "30_10-LQ-clustered.h5ad"
)
hq_adata.write(
    DATADIR / "processed" / "spatial" / "analysis" / "30_10-HQ-clustered.h5ad"
)

## Non Spatialdata

In [ ]:
samples = os.listdir(DATADIR / "spaceranger" / "LM13969")[1:]
print(samples)
adatas = [
    read_visium_hd_segmented(
        DATADIR / "spaceranger" / "LM13969" / sample / "outs", sample_id=sample
    )
    for sample in tqdm(samples)
]

In [ ]:
# calculated QCs
for n, ad in enumerate(adatas):
    sc.pp.calculate_qc_metrics(ad, inplace=True, percent_top=[20])
    Filter_GeneGroup(ad)
    ad.obs["Identifier"] = samples[n]

metadata = pd.concat([i.obs for i in adatas])
groupby = "Identifier"
check_QCPlot(metadata, "log1p_total_counts", groupby)
check_QCPlot(metadata, "n_genes_by_counts", groupby)
check_QCPlot(metadata, "pct_counts_in_top_20_genes", groupby)

metadata

In [ ]:
runFilters = True
processed = []

# QC Thresholds
filters = {"GenesPerCell": 0, "ReadsPerCell": 100, "CellsPerGene": 10, "Mito%": 15}

for n, ad in enumerate(adatas):
    adata = ad.copy()

    # Standard QC Filters
    Filter_QC(
        adata,
        GenePerCell=filters["GenesPerCell"],
        CountPerCell=filters["ReadsPerCell"],
        CellPerGene=filters["CellsPerGene"],
        verbose=True,
    )
    Filter_GeneGroup(
        adata,
        key="mito",
        marker="mt",
        verbose=True,
        perc_threshold=filters["Mito%"],
    )

    print(f"Remaining bins: {adata.shape[0]}\n")

    processed.append(adata)

adata_FULL = sc.concat(processed)

groupby = "Identifier"
check_QCPlot(adata_FULL.obs, "log1p_total_counts", groupby)
check_QCPlot(adata_FULL.obs, "n_genes_by_counts", groupby)
check_QCPlot(adata_FULL.obs, "pct_counts_in_top_20_genes", groupby)
adata_FULL

In [ ]:
# clean mitochondrial genes
adata_FULL.var["mito"] = adata_FULL.var_names.str.startswith("mt-")
adata_FULL.obsm["mito"] = adata_FULL[:, adata_FULL.var["mito"].values].X.toarray()
adata_FULL = adata_FULL[:, ~adata_FULL.var["mito"].values]

# clean Ig genes
adata_FULL.var["igs"] = adata_FULL.var_names.str.contains("Igh|Igk|Jchain")
adata_FULL.obsm["igs"] = adata_FULL[:, adata_FULL.var["igs"].values].X.toarray()
adata_FULL = adata_FULL[:, ~adata_FULL.var["igs"].values]

In [ ]:
batch_column = "Identifier"
adata_FULL.layers["counts"] = adata_FULL.X.copy()
Normalize(adata_FULL)
# FindVariableGenes(adata_FULL, "seurat_v3_paper", batch_column)
Integrate(adata_FULL, batch_column, False)
Visualize(adata_FULL, localmap=True)
sc.pl.embedding(adata_FULL, "UMAP")

In [ ]:
Cluster(adata_FULL, resolutions=[0.1, 0.5, 0.7, 1.0, 1.2, 1.5])

In [ ]:
f, ax = plt.subplots()
sc.pl.embedding(adata_FULL, "UMAP", color="leiden_1.2", ax=ax, show=False)
ax.annotate(
    f"n = {adata_FULL.shape[0]}",
    size=10,
    fontweight="bold",
    xy=(0.98, 0.02),
    xycoords="axes fraction",
    horizontalalignment="right",
    verticalalignment="bottom",
)

In [ ]:
clusters = "leiden_1.2"
de_key = "DEGs"

sc.tl.rank_genes_groups(
    adata_FULL,
    groupby=clusters,
    key_added=de_key,
    use_raw=False,
    layer="normalized",
    method="wilcoxon",
)

sc.pl.rank_genes_groups_dotplot(
    adata_FULL,
    groupby=clusters,
    key=de_key,
    standard_scale="var",
    n_genes=20,
    # min_logfoldchange=2,
    # ax=ax,
)

sc.pl.rank_genes_groups_heatmap(
    adata_FULL, key=de_key, groupby=[clusters], layer="normalized"
)

In [ ]:
adata_FULL.write(
    os.path.join(
        DATADIR, "processed", "spatial", "combined", "segmented-V1-200reads.h5ad"
    )
)